In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

RESULTS_DIR = '/content/drive/MyDrive/GAMEEMO_processed/results'
FIGURES_DIR = '/content/drive/MyDrive/GAMEEMO_processed/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

label_names = ['boring', 'calm', 'horror', 'funny']

with open(os.path.join(RESULTS_DIR, 'day2_results.pkl'), 'rb') as f:
    day2 = pickle.load(f)

with open(os.path.join(RESULTS_DIR, 'day3_results.pkl'), 'rb') as f:
    day3 = pickle.load(f)

with open(os.path.join(RESULTS_DIR, 'day4_results.pkl'), 'rb') as f:
    day4 = pickle.load(f)

# print exact keys so we can see what was saved
print('day2 keys:', list(day2.keys()))
print('day3 keys:', list(day3.keys()))
print('day4 keys:', list(day4.keys()))

day2 keys: ['eegnet_subject_dependent', 'eegnet_cross_subject', 'cnn1d_subject_dependent', 'cnn1d_cross_subject']
day3 keys: ['cnn_lstm_subject_dependent', 'cnn_lstm_cross_subject', 'cnn_bigru_subject_dependent', 'cnn_bigru_cross_subject']
day4 keys: ['transformer_subject_dependent', 'transformer_cross_subject']


In [ ]:
# day4 saved keys as 'transformer_subject_dependent' and 'transformer_cross_subject'
# but day5 was looking for 'eeg_transformer_subject_dependent' etc.
# we remap them here so everything is consistent

day4_fixed = {}
for k, v in day4.items():
    new_key = k.replace('transformer_', 'eeg_transformer_')
    day4_fixed[new_key] = v
    print('remapped:', k, '->', new_key)

all_results = {**day2, **day3, **day4_fixed}
print()
print('all keys after merge:', list(all_results.keys()))

remapped: transformer_subject_dependent -> eeg_transformer_subject_dependent
remapped: transformer_cross_subject -> eeg_transformer_cross_subject

all keys after merge: ['eegnet_subject_dependent', 'eegnet_cross_subject', 'cnn1d_subject_dependent', 'cnn1d_cross_subject', 'cnn_lstm_subject_dependent', 'cnn_lstm_cross_subject', 'cnn_bigru_subject_dependent', 'cnn_bigru_cross_subject', 'eeg_transformer_subject_dependent', 'eeg_transformer_cross_subject']


In [ ]:
model_order = [
    ('eegnet', 'EEGNet'),
    ('cnn1d', '1D-CNN'),
    ('cnn_lstm', 'CNN-LSTM'),
    ('cnn_bigru', 'CNN-BiGRU'),
    ('eeg_transformer', 'EEG Transformer'),
]

print(f"{'Model':<20} {'SD Acc':>10} {'SD F1':>10} {'CS Acc':>10} {'CS F1':>10}")
print('-' * 62)

for key, name in model_order:
    sd = all_results.get(f'{key}_subject_dependent', {})
    cs = all_results.get(f'{key}_cross_subject', {})
    sd_acc = round(sd.get('accuracy', 0) * 100, 2)
    sd_f1  = round(sd.get('f1_score', 0) * 100, 2)
    cs_acc = round(cs.get('accuracy', 0) * 100, 2)
    cs_f1  = round(cs.get('f1_score', 0) * 100, 2)
    print(f"{name:<20} {sd_acc:>9.2f}% {sd_f1:>9.2f}% {cs_acc:>9.2f}% {cs_f1:>9.2f}%")

Model                    SD Acc      SD F1     CS Acc      CS F1
--------------------------------------------------------------
EEGNet                   44.12%     43.80%     60.75%     60.29%
1D-CNN                   74.62%     74.97%     98.25%     98.25%
CNN-LSTM                 67.45%     68.69%     95.66%     95.68%
CNN-BiGRU                68.45%     69.63%     97.02%     97.02%
EEG Transformer          78.32%     78.87%     89.02%     88.98%


In [ ]:
model_names = [name for _, name in model_order]
sd_accs = []
cs_accs = []

for key, name in model_order:
    sd = all_results.get(f'{key}_subject_dependent', {})
    cs = all_results.get(f'{key}_cross_subject', {})
    sd_accs.append(sd.get('accuracy', 0) * 100)
    cs_accs.append(cs.get('accuracy', 0) * 100)

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, sd_accs, width, label='Subject-Dependent', color='steelblue')
bars2 = ax.bar(x + width/2, cs_accs, width, label='Cross-Subject', color='darkorange')

ax.set_ylabel('Accuracy (%)')
ax.set_title('GAMEEMO 4-Class Emotion Recognition - Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'accuracy_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved to', save_path)

saved to /content/drive/MyDrive/GAMEEMO_processed/figures/accuracy_comparison.png


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for idx, (key, name) in enumerate(model_order):
    sd = all_results.get(f'{key}_subject_dependent', {})
    cm = np.array(sd.get('confusion_matrix', np.zeros((4,4), dtype=int)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('')

fig.suptitle('Confusion Matrices - Subject-Dependent', fontsize=13, y=1.02)
plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'confusion_matrices_sd.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved to', save_path)

saved to /content/drive/MyDrive/GAMEEMO_processed/figures/confusion_matrices_sd.png


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for idx, (key, name) in enumerate(model_order):
    cs = all_results.get(f'{key}_cross_subject', {})
    cm = np.array(cs.get('confusion_matrix', np.zeros((4,4), dtype=int)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=axes[idx], colorbar=False, cmap='Oranges')
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('')

fig.suptitle('Confusion Matrices - Cross-Subject', fontsize=13, y=1.02)
plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'confusion_matrices_cs.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved to', save_path)

saved to /content/drive/MyDrive/GAMEEMO_processed/figures/confusion_matrices_cs.png


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for idx, (key, name) in enumerate(model_order):
    for row, protocol in enumerate(['subject_dependent', 'cross_subject']):
        res = all_results.get(f'{key}_{protocol}', {})
        history = res.get('history', {})
        ax = axes[row][idx]
        if 'accuracy' in history:
            ax.plot(history['accuracy'], label='train')
        if 'val_accuracy' in history:
            ax.plot(history['val_accuracy'], label='val')
        ax.set_title(name, fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.grid(True, linestyle='--', alpha=0.4)
        if idx == 0:
            protocol_label = 'Subject-Dep' if protocol == 'subject_dependent' else 'Cross-Subj'
            ax.set_ylabel(protocol_label)
        if row == 0:
            ax.legend(fontsize=7)

fig.suptitle('Training Curves - Accuracy', fontsize=13)
plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'training_curves.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved to', save_path)

saved to /content/drive/MyDrive/GAMEEMO_processed/figures/training_curves.png


In [ ]:
print('all figures saved to:', FIGURES_DIR)
print()
print('final summary:')
print(f"{'Model':<20} {'SD Acc':>10} {'CS Acc':>10}")
print('-' * 42)
for key, name in model_order:
    sd_acc = all_results.get(f'{key}_subject_dependent', {}).get('accuracy', 0) * 100
    cs_acc = all_results.get(f'{key}_cross_subject', {}).get('accuracy', 0) * 100
    print(f"{name:<20} {sd_acc:>9.2f}% {cs_acc:>9.2f}%")

all figures saved to: /content/drive/MyDrive/GAMEEMO_processed/figures

final summary:
Model                    SD Acc     CS Acc
------------------------------------------
EEGNet                   44.12%     60.75%
1D-CNN                   74.62%     98.25%
CNN-LSTM                 67.45%     95.66%
CNN-BiGRU                68.45%     97.02%
EEG Transformer          78.32%     89.02%


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, (key, name) in enumerate(model_order):

    cs = all_results.get(f'{key}_cross_subject', {})

    cm = np.array(
        cs.get('confusion_matrix', np.zeros((4, 4), dtype=int))
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_names
    )

    disp.plot(
        ax=axes[idx],
        colorbar=False,
        cmap='Oranges'
    )

    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('')

axes[5].axis('off')

fig.suptitle(
    'Confusion Matrices - Cross-Subject',
    fontsize=13,
    y=1.02
)

plt.tight_layout()

save_path = os.path.join(
    FIGURES_DIR,
    'confusion_matrices_cs.png'
)

plt.savefig(
    save_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()

print('saved to', save_path)

saved to /content/drive/MyDrive/GAMEEMO_processed/figures/confusion_matrices_cs.png


In [ ]:
fig, axes = plt.subplots(
    2, 5,
    figsize=(22, 8)
)

protocols = ['subject_dependent', 'cross_subject']

for idx, (key, name) in enumerate(model_order):

    for row, protocol in enumerate(protocols):

        res = all_results.get(f'{key}_{protocol}', {})
        history = res.get('history', {})
        ax = axes[row, idx]


        if 'loss' in history:
            ax.plot(
                history['loss'],
                label='Training',
                linewidth=1.5
            )


        if 'val_loss' in history:
            ax.plot(
                history['val_loss'],
                label='Validation',
                linewidth=1.5
            )

        ax.set_title(name, fontsize=10)
        ax.set_xlabel('Epoch', fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.4)


        if idx == 0:
            ax.set_ylabel(
                'Subject-Dependent Loss'
                if protocol == 'subject_dependent'
                else 'Cross-Subject Loss',
                fontsize=9
            )

        ax.legend(
            fontsize=7,
            loc='upper right'
        )

fig.suptitle(
    'Training and Validation Loss Curves Across Models',
    fontsize=14,
    y=1.02
)

plt.tight_layout()

save_path = os.path.join(
    FIGURES_DIR,
    'loss_curves.png'
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print('Saved to:', save_path)

Saved to: /content/drive/MyDrive/GAMEEMO_processed/figures/loss_curves.png


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
axes = axes.flatten()

subplot_labels = ['(a)', '(b)', '(c)', '(d)', '(e)']

for idx, (key, name) in enumerate(model_order):

    cs = all_results.get(f'{key}_cross_subject', {})

    cm = np.array(
        cs.get('confusion_matrix', np.zeros((4, 4), dtype=int))
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_names
    )

    disp.plot(
        ax=axes[idx],
        colorbar=False,
        cmap='Oranges'
    )


    axes[idx].set_title(name, fontsize=11, pad=8)

    axes[idx].text(
        -0.15, 1.08,
        subplot_labels[idx],
        transform=axes[idx].transAxes,
        fontsize=12,
        fontweight='bold',
        va='top',
        ha='right'
    )

    axes[idx].set_xlabel('Predicted Label', fontsize=9)
    axes[idx].set_ylabel('True Label', fontsize=9)

axes[5].axis('off')

fig.suptitle(
    'Confusion Matrices for Cross-Subject Emotion Classification',
    fontsize=14,
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])

save_path = os.path.join(
    FIGURES_DIR,
    'confusion_matrices_cross_subject.png'
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print('Saved to:', save_path)

Saved to: /content/drive/MyDrive/GAMEEMO_processed/figures/confusion_matrices_cross_subject.png


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
axes = axes.flatten()

subplot_labels = ['(a)', '(b)', '(c)', '(d)', '(e)']

for idx, (key, name) in enumerate(model_order):

    cs = all_results.get(f'{key}_cross_subject', {})

    cm = np.array(
        cs.get('confusion_matrix', np.zeros((4, 4), dtype=int))
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_names
    )

    disp.plot(
        ax=axes[idx],
        colorbar=False,
        cmap='Oranges'
    )

    axes[idx].set_title('')

    axes[idx].text(
        0.5, -0.20,
        f'{subplot_labels[idx]} {name}',
        transform=axes[idx].transAxes,
        fontsize=11,
        fontweight='bold',
        ha='center',
        va='top'
    )

    axes[idx].set_xlabel('Predicted Label', fontsize=9)
    axes[idx].set_ylabel('True Label', fontsize=9)

axes[5].axis('off')

fig.suptitle(
    'Confusion Matrices for Cross-Subject Emotion Classification',
    fontsize=14,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0.02, 1, 0.97])

save_path = os.path.join(
    FIGURES_DIR,
    'confusion_matrices_cross_subject.png'
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print('Saved to:', save_path)

Saved to: /content/drive/MyDrive/GAMEEMO_processed/figures/confusion_matrices_cross_subject.png
